# 大模型量化深度实验 - 综合教学Lab

**作者**: AI Research Lab  
**难度**: ⭐⭐⭐⭐ (中高级)  
**预计时间**: 8-12小时

## 🎯 实验目标

通过本实验，你将：
1. 从零实现量化算法的核心组件
2. 理解对称/非对称、逐张量/逐通道量化的区别
3. 掌握PTQ和QAT的原理与实现
4. 学习GPTQ等高级量化技术
5. 分析量化对模型性能的影响
6. 优化量化策略以平衡精度和压缩率

## 📋 实验流程

1. **Part 1**: 量化基础 (1-2小时)
2. **Part 2**: PTQ训练后量化 (2-3小时)
3. **Part 3**: QAT量化感知训练 (2-3小时)
4. **Part 4**: GPTQ高级量化 (2-3小时)
5. **Part 5**: 综合评估与优化 (1-2小时)

## ⚙️ 环境检查

In [ ]:
# 导入必要的库
import sys
import os

# 添加项目路径
sys.path.insert(0, os.path.abspath('.'))

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# 设置绘图风格
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# 检查CUDA
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
print(f"PyTorch版本: {torch.__version__}")

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU内存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

---
# Part 1: 量化基础 (Quantization Basics)

## 1.1 量化理论

### 什么是量化？

量化是将高精度浮点数映射到低精度整数的过程：

$$
x_{\text{quant}} = \text{round}\left(\frac{x_{\text{float}}}{\text{scale}}\right) + \text{zero\_point}
$$

### 对称量化 vs 非对称量化

**对称量化**:
- $\text{zero\_point} = 0$
- $\text{scale} = \frac{\max(|x|)}{2^{n-1}-1}$
- 量化范围: $[-2^{n-1}, 2^{n-1}-1]$

**非对称量化**:
- $\text{scale} = \frac{\max(x) - \min(x)}{2^n - 1}$
- $\text{zero\_point} = -\text{round}(\frac{\min(x)}{\text{scale}})$
- 量化范围: $[0, 2^n-1]$

### 逐张量 vs 逐通道量化

- **逐张量**: 整个张量共享一个scale和zero_point
- **逐通道**: 每个输出通道有独立的scale和zero_point（更精确）

## 1.2 实现基础量化函数

**学生任务**: 完成核心量化函数（已在`core/quantization_basics.py`中）

In [ ]:
from core.quantization_basics import (
    QuantizationConfig,
    BasicQuantizer,
    compute_quantization_error,
)

# 测试基础量化
print("=" * 80)
print("测试1: 对称量化 vs 非对称量化")
print("=" * 80)

# 创建测试数据（模拟权重）
weight = torch.randn(128, 256)
print(f"\n原始权重统计:")
print(f"  均值: {weight.mean():.6f}")
print(f"  标准差: {weight.std():.6f}")
print(f"  范围: [{weight.min():.6f}, {weight.max():.6f}]")

# 对称量化
config_sym = QuantizationConfig(n_bits=8, symmetric=True, per_channel=False)
quantizer_sym = BasicQuantizer(config_sym)
quantizer_sym.calibrate(weight)

quantized_sym = quantizer_sym.quantize(weight)
dequantized_sym = quantizer_sym.dequantize(quantized_sym)
errors_sym = compute_quantization_error(weight, quantized_sym, dequantized_sym)

print(f"\n对称量化 (8-bit):")
print(f"  Scale: {quantizer_sym.scale.item():.8f}")
print(f"  SQNR: {errors_sym['sqnr_db']:.2f} dB")
print(f"  余弦相似度: {errors_sym['cosine_similarity']:.6f}")

# 非对称量化
config_asym = QuantizationConfig(n_bits=8, symmetric=False, per_channel=False)
quantizer_asym = BasicQuantizer(config_asym)
quantizer_asym.calibrate(weight)

quantized_asym = quantizer_asym.quantize(weight)
dequantized_asym = quantizer_asym.dequantize(quantized_asym)
errors_asym = compute_quantization_error(weight, quantized_asym, dequantized_asym)

print(f"\n非对称量化 (8-bit):")
print(f"  Scale: {quantizer_asym.scale.item():.8f}")
print(f"  Zero-point: {quantizer_asym.zero_point.item()}")
print(f"  SQNR: {errors_asym['sqnr_db']:.2f} dB")
print(f"  余弦相似度: {errors_asym['cosine_similarity']:.6f}")

## 1.3 可视化量化效果

**学生任务**: 可视化不同量化配置的效果

In [ ]:
# ==================== YOUR CODE HERE (开始) ====================
# TODO: 绘制量化前后的分布对比图

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 原始分布
axes[0, 0].hist(weight.flatten().numpy(), bins=100, alpha=0.7, color='blue')
axes[0, 0].set_title('原始权重分布', fontsize=14)
axes[0, 0].set_xlabel('值')
axes[0, 0].set_ylabel('频数')
axes[0, 0].grid(True, alpha=0.3)

# 对称量化后
axes[0, 1].hist(dequantized_sym.flatten().numpy(), bins=100, alpha=0.7, color='orange')
axes[0, 1].set_title(f'对称量化后 (SQNR={errors_sym["sqnr_db"]:.1f}dB)', fontsize=14)
axes[0, 1].set_xlabel('值')
axes[0, 1].set_ylabel('频数')
axes[0, 1].grid(True, alpha=0.3)

# 非对称量化后
axes[1, 0].hist(dequantized_asym.flatten().numpy(), bins=100, alpha=0.7, color='green')
axes[1, 0].set_title(f'非对称量化后 (SQNR={errors_asym["sqnr_db"]:.1f}dB)', fontsize=14)
axes[1, 0].set_xlabel('值')
axes[1, 0].set_ylabel('频数')
axes[1, 0].grid(True, alpha=0.3)

# 量化误差分布
error_sym = (weight - dequantized_sym).flatten().numpy()
error_asym = (weight - dequantized_asym).flatten().numpy()
axes[1, 1].hist(error_sym, bins=100, alpha=0.6, color='orange', label='对称')
axes[1, 1].hist(error_asym, bins=100, alpha=0.6, color='green', label='非对称')
axes[1, 1].set_title('量化误差分布', fontsize=14)
axes[1, 1].set_xlabel('误差')
axes[1, 1].set_ylabel('频数')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ==================== YOUR CODE HERE (结束) ====================

## 1.4 不同位宽的影响

**学生任务**: 比较2-bit、4-bit、8-bit量化的效果

In [ ]:
# ==================== YOUR CODE HERE (开始) ====================
# TODO: 比较不同位宽的量化效果

bit_widths = [2, 3, 4, 6, 8, 16]
results = {'bits': [], 'sqnr': [], 'cosine_sim': [], 'compression': []}

for n_bits in bit_widths:
    config = QuantizationConfig(n_bits=n_bits, symmetric=True, per_channel=False)
    quantizer = BasicQuantizer(config)
    quantizer.calibrate(weight)
    
    quantized = quantizer.quantize(weight)
    dequantized = quantizer.dequantize(quantized)
    errors = compute_quantization_error(weight, quantized, dequantized)
    
    results['bits'].append(n_bits)
    results['sqnr'].append(errors['sqnr_db'])
    results['cosine_sim'].append(errors['cosine_similarity'])
    results['compression'].append(quantizer.get_compression_ratio())

# 绘制结果
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# SQNR vs 位宽
axes[0].plot(results['bits'], results['sqnr'], marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('位宽 (bits)', fontsize=12)
axes[0].set_ylabel('SQNR (dB)', fontsize=12)
axes[0].set_title('信噪比 vs 位宽', fontsize=14)
axes[0].grid(True, alpha=0.3)
axes[0].set_xticks(results['bits'])

# 余弦相似度 vs 位宽
axes[1].plot(results['bits'], results['cosine_sim'], marker='s', linewidth=2, markersize=8, color='orange')
axes[1].set_xlabel('位宽 (bits)', fontsize=12)
axes[1].set_ylabel('余弦相似度', fontsize=12)
axes[1].set_title('余弦相似度 vs 位宽', fontsize=14)
axes[1].grid(True, alpha=0.3)
axes[1].set_xticks(results['bits'])
axes[1].set_ylim([0.9, 1.0])

# 压缩率 vs 位宽
axes[2].bar(results['bits'], results['compression'], color='green', alpha=0.7)
axes[2].set_xlabel('位宽 (bits)', fontsize=12)
axes[2].set_ylabel('压缩率', fontsize=12)
axes[2].set_title('压缩率 vs 位宽', fontsize=14)
axes[2].grid(True, alpha=0.3, axis='y')
axes[2].set_xticks(results['bits'])

plt.tight_layout()
plt.show()

print("\n位宽比较结果:")
for i, bits in enumerate(results['bits']):
    print(f"{bits}-bit: SQNR={results['sqnr'][i]:.2f}dB, "
          f"Cosine={results['cosine_sim'][i]:.6f}, "
          f"压缩={results['compression'][i]:.1f}x")

# ==================== YOUR CODE HERE (结束) ====================

**思考题**:
1. 为什么SQNR随位宽增加而提高？量化关系是什么？
2. 什么情况下对称量化优于非对称量化？
3. 逐通道量化相比逐张量量化有什么优势？

---
# Part 2: PTQ - 训练后量化

## 2.1 PTQ原理

训练后量化(PTQ)是在已训练好的模型上直接进行量化，无需重新训练。关键步骤：

1. **校准** (Calibration): 使用少量数据确定量化参数
2. **量化**: 将权重和激活转换为低精度
3. **评估**: 测试量化后的精度损失

### 校准方法

- **MinMax**: 使用观察到的最小/最大值
- **Percentile**: 使用百分位数（忽略离群值）
- **MSE**: 搜索最小化量化误差的参数
- **Entropy**: 最小化KL散度

In [ ]:
from core.ptq_static import (
    MinMaxCalibration,
    PercentileCalibration,
    MSECalibration,
    PTQQuantizer,
)

# 创建一个简单的测试模型
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 128)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.fc3(x)
        return x

model = SimpleNet()
print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
print(f"模型大小: {sum(p.numel() for p in model.parameters()) * 4 / 1e6:.2f} MB (FP32)")

## 2.2 比较不同校准方法

**学生任务**: 实现并比较MinMax、Percentile、MSE三种校准方法

In [ ]:
# ==================== YOUR CODE HERE (开始) ====================
# TODO: 比较不同校准方法

# 生成模拟激活数据（包含离群值）
torch.manual_seed(42)
activations = []
for i in range(20):
    act = torch.randn(32, 256) * 0.5
    # 添加一些离群值
    if i % 5 == 0:
        act[0, 0] = 10.0  # 极端离群值
    activations.append(act)

config = QuantizationConfig(n_bits=8, symmetric=False, per_channel=False)

methods = {
    'MinMax': MinMaxCalibration(config),
    'Percentile-99.9': PercentileCalibration(config, percentile=99.9),
    'Percentile-99.99': PercentileCalibration(config, percentile=99.99),
    'MSE': MSECalibration(config, num_candidates=50),
}

results = {}
for name, calibrator in methods.items():
    # 收集统计
    for act in activations:
        calibrator.collect_stats(act)
    
    # 计算量化参数
    scale, zero_point = calibrator.compute_qparams()
    
    # 测试量化效果
    test_act = torch.cat(activations, dim=0)
    quantized = quantize_tensor(test_act, scale, zero_point, config.quant_min, config.quant_max)
    dequantized = dequantize_tensor(quantized, scale, zero_point)
    
    errors = compute_quantization_error(test_act, quantized, dequantized)
    
    results[name] = {
        'scale': scale.item(),
        'zero_point': zero_point.item(),
        'sqnr': errors['sqnr_db'],
        'mse': errors['mse'],
    }

# 打印结果
print("\n校准方法比较:")
print("-" * 80)
print(f"{'方法':<20} {'Scale':<15} {'Zero-point':<12} {'SQNR (dB)':<12} {'MSE'}")
print("-" * 80)
for name, res in results.items():
    print(f"{name:<20} {res['scale']:<15.6f} {res['zero_point']:<12} {res['sqnr']:<12.2f} {res['mse']:.6f}")

# ==================== YOUR CODE HERE (结束) ====================